# Phase D: MATLAB Viewer Controlled API Validation

This notebook validates the new `viewer_ctrl` additive facade which allows Python to natively control the validated MATLAB AWR2944 dashboard via COM automation, without regenerating data or modifying validated scripts.

It strictly tests offline accepted captures with zero hardware access.

In [ ]:
from pathlib import Path
from awr2944_dca.lab import RadarProject

PROJECT_ROOT = Path(r"C:\Users\khams008\Documents\awr2944-live-project")
project = RadarProject.open(PROJECT_ROOT)

# List accepted captures
captures = project.captures.list()
print("Available Captures:")
for c in captures:
    print(f"- {c.capture_id} ({c.status})")

# Explicitly select an accepted capture (replace with actual ID if running live)
# Select the last complete capture explicitly
complete = [c for c in captures if c.status().get('status') == 'complete']
if not complete:
    raise RuntimeError("No complete captures found. Please ensure at least one capture has passed hardware validation.")

CAPTURE_ID = None  # Set to "explicit_id" to force a specific capture

if CAPTURE_ID:
    capture_id = CAPTURE_ID
else:
    complete = [c for c in captures if c.status().get('status') == 'complete']
    if not complete:
        raise RuntimeError("No complete captures found.")
    capture_id = complete[-1].capture_id
capture = project.captures.get(capture_id)
capture.verify(strict=True)
print(f"\nSelected Capture: {capture.capture_id}")

### 1. Existing Accepted Capture Verification

In [ ]:
report = capture.verify(strict=True)
print("Verification Status:", "PASS" if report.success else "FAIL")

### 2. Existing Raw ADC Public Access

In [ ]:
raw = capture.raw
print("Native Path:", raw.native_path)
print("Native Bytes:", raw.native_bytes)
print("Canonical Path:", raw.canonical_path)
print("Canonical Bytes:", raw.canonical_bytes)

# Check frames iterator shape without loading the whole cube to RAM unnecessarily
frame_count = 0
last_shape = None
for frame_idx, frame_data in enumerate(raw.iter_frames()):
    frame_count += 1
    last_shape = frame_data.shape
print(f"\nIterated {frame_count} frames. Shape per frame: {last_shape}")

### 3. Existing Viewer-Payload Read-Only Access
Verify the exported metadata and arrays are intact without invoking DSP or recalculating.

In [ ]:
vd = capture.viewer_data
print(f"Viewer Payload Exists: {vd.exists}")
print(f"Payload Path: {vd.path}")
if vd.exists:
    payload = vd.load()
    print("\nPayload Keys Loaded successfully via SciPy.")
    print(f"Frame Count in Payload: {payload.get('frame_count')}")
    if 'range_axis_m' in payload:
        print(f"Range Axis Shape: {payload['range_axis_m'].shape}")

### 4. Controlled MATLAB Dashboard Launch & UI Extraction

In [ ]:
import time
import IPython.display as display

with capture.open_controlled_viewer() as viewer:
    print(f"Spawned MATLAB COM Session with token: {viewer.token}")
    viewer.wait_ready(timeout=30.0)
    print("Viewer is ready.")
    
    plots = viewer.list_plots()
    print(f"\nAvailable Plots: {plots}")
    
    # Test initial frame
    frame_0 = viewer.get_frame()
    print(f"Initial Frame: {frame_0}")
    
    # Frame change through callback
    print("\nSetting to frame 5...")
    viewer.set_frame(5)
    current_frame = viewer.get_frame()
    print(f"Current Frame is now: {current_frame}")
    
    # Retrieval of displayed FFT/plot data directly from MATLAB graphics objects
    print("\nExtracting Range-Doppler plot properties directly from the active viewer axis...")
    rd_data = viewer.get_displayed_plot("range_doppler")
    print(f"  Type: {rd_data.get('type')}")
    print(f"  Title: {rd_data.get('Title')}")
    if 'CData' in rd_data:
        import numpy as np
        cdata = np.array(rd_data['CData'])
        print(f"  CData Extracted Shape: {cdata.shape}")
        print(f"  CLim: {rd_data.get('CLim')}")
        
    # Automated graphics export is explicitly unsupported to prevent COM deadlocks.
    # Attempts to call export methods will raise a ViewerExportUnsupportedError:
    from awr2944_dca.viewer_ctrl import ViewerExportUnsupportedError
    try:
        viewer.export_plot("range_doppler")
    except ViewerExportUnsupportedError as e:
        print(f"\nCaught expected export exception: {e}")
        
        
# Outside the block, the context manager calls viewer.close() automatically.
print("\nViewer automatically closed.")

### 5. Manual Figure Export and Screenshots

Automated COM-driven graphics export via `exportgraphics`, `print`, or `saveas` within the background timer callback fundamentally deadlocks the MATLAB engine event queue on Windows.

**To export figures:**
1. Keep the `controlled_viewer` context active (or use `open_viewer()` for an unmanaged session).
2. Use the **MATLAB Dashboard GUI** manually:
   - Navigate to `File -> Save As` to save a `.fig`, `.pdf`, or `.svg`.
   - Or use standard OS screenshot tools (e.g., Windows Snipping Tool) for raster captures.

### 6. Close and Relaunch
Ensure the COM architecture is isolated and handles a relaunch smoothly.

In [ ]:
print("Relaunching viewer...")
with capture.open_controlled_viewer() as viewer2:
    print(f"Spawned 2nd MATLAB COM Session with token: {viewer2.token}")
    viewer2.wait_ready(timeout=30.0)
    viewer2.set_frame(10)
    print(f"Set Frame: {viewer2.get_frame()}")
    viewer2.close()
print("Relaunch test passed.")